# RAG adım adım: paperlens içinden bir soru

Bu defter tek bir soruyu RAG'in her aşamasından geçirir ve her aşamanın **gerçek çıktısını** gösterir:

1. Veri → 2. Chunking → 3. Embedding → 4. Vektör DB → 5. BM25 → 6. Hybrid (RRF) → 7. Mini retrieval eval → 8. Prompt → 9. LLM → 10. Puanlama

**Nasıl kullanılır:** hücreleri yukarıdan aşağıya sırayla çalıştır (`Shift+Enter`). Büyük harfle yazılmış değişkenler (`QUESTION_INDEX`, `CHUNK_SIZE`, `RRF_K`, `TOP_K`…) senin değiştirmen için; değiştirip o hücreyi ve sonrakileri tekrar çalıştır.

**Gerekenler:** GPU (bge-m3 için) ve 9. adım için çalışan Ollama (`systemctl is-active ollama`). Kod `src/paperlens/` içindeki gerçek proje kodunu kullanır; burada yeniden yazılmış bir şey yoktur.

## 0 · Kurulum

Config, tüm parametrelerin tek kaynağıdır. Defter `experiments/configs/baseline.yaml` dosyasını okur; deneylerdeki her şey buradan gelir.

In [ ]:
import os
from pathlib import Path

# Defter notebooks/ klasöründe; config'deki göreli yollar (data/, prompts/) proje kökünden çalışır
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

import matplotlib.pyplot as plt
import numpy as np

from paperlens.config.loader import load_config

cfg = load_config("experiments/configs/baseline.yaml")
print("chunking :", cfg.chunking)
print("embedding:", cfg.embedding.model_name)
print("retrieval:", cfg.retrieval)
print("llm      :", cfg.llm.model)

## 1 · Veri: makale ve soru

`ingestion/qasper.py` QASPER'i indirir (ilk seferde) ve her satırı `Paper` / `Question` nesnelerine çevirir. Her paragrafın kalıcı bir id'si vardır (`<makale>::s<bölüm>p<paragraf>`); gold kanıtlar bu id'lere eşlenir, retrieval da bu id'lerle puanlanır.

`QUESTION_INDEX`, deneylerde kullanılan 250 soruluk örneklemdeki sıradır (0–249). 16 = SnapCaptions sorusu (dokümandaki örnek).

In [ ]:
from paperlens.ingestion.qasper import download_split, load_split, sample_questions

papers, questions = load_split(download_split(cfg.data, cfg.data.split))
sample = sample_questions(questions, cfg.data.sample_size, cfg.data.seed)
print(f"{len(papers)} makale, {len(questions)} soru, örneklem {len(sample)} soru")

QUESTION_INDEX = 16  # <- değiştir (0-249)
question = sample[QUESTION_INDEX]
paper = next(p for p in papers if p.paper_id == question.paper_id)
gold_ids = {pid for r in question.references for pid in r.evidence_ids}

print("\nSoru  :", question.question)
print("Makale:", paper.paper_id, "-", paper.title)
for i, r in enumerate(question.references, 1):
    print(f"Gold {i} ({r.answer_type}): {r.answer_text}  | kanıt: {r.evidence_ids}")

In [ ]:
# Makalenin ilk paragrafları: id, bölüm, metin başı
for p in paper.paragraphs[:12]:
    mark = "  <- GOLD" if p.paragraph_id in gold_ids else ""
    print(f"{p.paragraph_id:24} {p.section[:28]:28} {p.text[:60]!r}{mark}")
print(f"... toplam {len(paper.paragraphs)} paragraf")

## 2 · Chunking

`ingestion/chunkers.py` → `chunk_paper(paper, cfg)` makaleyi parçalara böler ve her parça hangi paragraflardan oluştuğunu `paragraph_ids` içinde saklar.

- `fixed`: tüm kelimeleri sıraya dizer, `CHUNK_SIZE` kelimelik pencereyi `CHUNK_SIZE - OVERLAP` kadar kaydırır. Bölüm sınırına bakmaz.
- `section`: bütün paragrafları `CHUNK_SIZE`'ı aşmayacak şekilde paketler, bölüm sınırını geçmez.

**Dene:** `STRATEGY = "section"` yap ve gold chunk'ın kaç paragraf içerdiğine bak. Sonra `CHUNK_SIZE = 100` ve `400` dene.

In [ ]:
from paperlens.ingestion.chunkers import chunk_paper

STRATEGY = "fixed"  # <- "fixed" veya "section"
CHUNK_SIZE = 200  # <- kelime
OVERLAP = 40  # <- kelime (sadece fixed için)

ccfg = cfg.chunking.model_copy(
    update={"strategy": STRATEGY, "chunk_size": CHUNK_SIZE, "chunk_overlap": OVERLAP}
)
chunks = chunk_paper(paper, ccfg)
print(f"{len(paper.paragraphs)} paragraf -> {len(chunks)} chunk\n")
for c in chunks[:3]:
    print(c.chunk_id, f"{len(c.text.split())} kelime", c.paragraph_ids)
print("\nCevabı içeren chunk'lar:")
for c in chunks:
    if gold_ids & set(c.paragraph_ids):
        print(" ", c.chunk_id, repr(c.section), c.paragraph_ids)

In [ ]:
# Bir chunk'ın tam metni: başındaki [Bölüm adı] include_section_title ayarından gelir
gold_chunk = next(c for c in chunks if gold_ids & set(c.paragraph_ids))
print(gold_chunk.text)

In [ ]:
# Tüm 281 makalede chunk uzunluk dağılımı: fixed hep ~200, section çok değişken
fixed_lens = [len(c.text.split()) for p in papers for c in chunk_paper(p, cfg.chunking)]
section_cfg = cfg.chunking.model_copy(update={"strategy": "section"})
section_lens = [len(c.text.split()) for p in papers for c in chunk_paper(p, section_cfg)]
print(f"fixed: {len(fixed_lens)} chunk, section: {len(section_lens)} chunk")
plt.figure(figsize=(8, 3))
plt.hist([fixed_lens, section_lens], bins=40, label=["fixed", "section"])
plt.xlabel("chunk uzunluğu (kelime)")
plt.ylabel("chunk sayısı")
plt.legend()
plt.show()

## 3 · Embedding

`index/embedder.py` → `SentenceTransformerEmbedder` bge-m3'ü GPU'ya yükler. `embed_passages()` chunk'ları, `embed_query()` soruyu 1024 sayılık, uzunluğu 1 olan bir vektöre çevirir. Vektörler normalize olduğu için **cosine benzerliği = dot product** (`a @ b`).

İlk çalıştırmada model yüklenir (~10-20 sn).

In [ ]:
from paperlens.index.embedder import build_embedder

embedder = build_embedder(cfg.embedding)
qv = embedder.embed_query(question.question)
print("boyut:", qv.shape, "| uzunluk (norm):", round(float(np.linalg.norm(qv)), 4))
print("ilk 8 sayı:", np.round(qv[:8], 4))

In [ ]:
# Anlam benzerliği: ortak kelime olmadan bile eş anlamlılar yakın çıkar
SENTENCES = [  # <- kendi cümlelerini ekle
    "The car is fast.",
    "The automobile is quick.",
    "I like bananas.",
    question.question,
]
E = embedder.embed_passages(SENTENCES)
sim = E @ E.T
labels = [s[:22] for s in SENTENCES]
plt.figure(figsize=(5, 4))
plt.imshow(sim, vmin=0, vmax=1, cmap="viridis")
plt.xticks(range(len(labels)), labels, rotation=30, ha="right")
plt.yticks(range(len(labels)), labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        plt.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center", color="white")
plt.colorbar()
plt.title("cosine benzerliği")
plt.show()

In [ ]:
# Soru ile makaledeki HER chunk'ın benzerliği = elle yapılan 'flat' (brute force) arama
C = embedder.embed_passages([c.text for c in chunks])
scores = C @ qv
is_gold = [bool(gold_ids & set(c.paragraph_ids)) for c in chunks]
plt.figure(figsize=(10, 3))
plt.bar(range(len(chunks)), scores, color=["tab:orange" if g else "tab:blue" for g in is_gold])
plt.xlabel("chunk (makaledeki sıra)")
plt.ylabel("cosine")
plt.title("turuncu = cevabı içeren chunk")
plt.show()

for rank, i in enumerate(np.argsort(-scores)[:5], 1):
    print(rank, chunks[i].chunk_id, round(float(scores[i]), 4), "GOLD" if is_gold[i] else "")

## 4 · Vektör DB (Qdrant)

`index/store.py` → `build_or_load()` tüm makalelerin chunk'larını vektöre çevirip Qdrant'a yazar ve diske kaydeder; aynı ayarlarla ikinci seferde diskten yükler (`cache hit`). Anahtar sadece indeksi değiştiren ayarlardan hesaplanır.

Not: bu indeks config'deki chunking'i kullanır (2. adımdaki `STRATEGY`/`CHUNK_SIZE` değişikliklerin burayı etkilemez).

In [ ]:
from paperlens.index.store import build_or_load, index_key

bundle = build_or_load(cfg, papers, embedder)
print(
    "indeks anahtarı:",
    index_key(cfg),
    "| toplam chunk:",
    len(bundle.chunks),
    "| cache hit:",
    bundle.cache_hit,
)

In [ ]:
# Qdrant'ta saklanan bir 'point': id + vektör + payload (öğrenme amaçlı iç client'a erişiyoruz)
points, _ = bundle.dense._client.scroll("chunks", limit=1, with_vectors=True)
pt = points[0]
print("id     :", pt.id)
print("vektör :", len(pt.vector), "sayı, ilk 5:", np.round(pt.vector[:5], 4))
print(
    "payload:",
    {
        k: (v[:60] + "...") if isinstance(v, str) and len(v) > 60 else v
        for k, v in pt.payload.items()
    },
)

In [ ]:
# Aynı arama Qdrant ile: skorlar 3. adımdaki elle hesaplanan dot product'la aynı olmalı
hits = bundle.dense.search(question.question, k=5, paper_id=question.paper_id)
for rank, h in enumerate(hits, 1):
    print(
        rank,
        f"{h.score:.4f}",
        h.chunk.chunk_id,
        repr(h.chunk.section),
        "GOLD" if gold_ids & set(h.chunk.paragraph_ids) else "",
    )

# Metadata filtresi olmadan: tüm 281 makalede arama (corpus mode'un zorluğu)
print("\npaper_id filtresi OLMADAN:")
for h in bundle.dense.search(question.question, k=5):
    print(
        f"{h.score:.4f}",
        h.chunk.paper_id,
        "<- doğru makale" if h.chunk.paper_id == question.paper_id else "",
    )

## 5 · BM25 (kelime araması)

`index/sparse.py` → `SparseIndex`. Soru kelimelere ayrılır, stopword'ler atılır; chunk puanı kelime sıklığı, kelimenin nadirliği (IDF) ve chunk uzunluğundan gelir. Anlamı bilmez: "how large" ile "composed of 10K" eşleşmez.

In [ ]:
import bm25s

print(
    "soru kelimeleri:",
    bm25s.tokenize([question.question], stopwords="en", return_ids=False, show_progress=False)[0],
)
bm = bundle.sparse.search(question.question, k=5, paper_id=question.paper_id)
for rank, h in enumerate(bm, 1):
    print(
        rank,
        f"{h.score:.3f}",
        h.chunk.chunk_id,
        repr(h.chunk.section),
        "GOLD" if gold_ids & set(h.chunk.paragraph_ids) else "",
    )

## 6 · Hybrid: Reciprocal Rank Fusion

`retrieval/fusion.py` iki sıralamayı skorlarına değil **sıralarına** bakarak birleştirir: her chunk her listede `1 / (RRF_K + sıra)` puan alır, puanlar toplanır. Aşağıda kütüphanenin sonucu ile elle hesap yan yana.

**Dene:** `RRF_K = 10` ve `RRF_K = 200`; sıralama nasıl değişiyor?

In [ ]:
from paperlens.retrieval.fusion import reciprocal_rank_fusion

RRF_K = 60  # <- değiştir
CANDIDATE_K = 30
dense_c = bundle.dense.search(question.question, CANDIDATE_K, question.paper_id)
sparse_c = bundle.sparse.search(question.question, CANDIDATE_K, question.paper_id)
fused = reciprocal_rank_fusion([dense_c, sparse_c], k=RRF_K, top_n=5)

rank_d = {h.chunk.chunk_id: r for r, h in enumerate(dense_c, 1)}
rank_s = {h.chunk.chunk_id: r for r, h in enumerate(sparse_c, 1)}
print(f"{'chunk':18} {'dense':>5} {'bm25':>5} {'RRF':>8} {'elle':>8}")
for h in fused:
    cid = h.chunk.chunk_id
    manual = sum(1 / (RRF_K + r) for r in (rank_d.get(cid), rank_s.get(cid)) if r)
    gold = "GOLD" if gold_ids & set(h.chunk.paragraph_ids) else ""
    print(
        f"{cid:18} {rank_d.get(cid, '-'):>5} {rank_s.get(cid, '-'):>5} {h.score:8.5f} {manual:8.5f} {gold}"
    )

## 7 · Mini retrieval eval (LLM'siz, ~1 dk)

LLM'i hiç çağırmadan 250 sorunun hepsinde retrieval'ı ölçebilirsin: Recall@k = "ilk k chunk'tan en az biri gold paragrafı içeriyor mu". Bu, retrieval ayarlarını denemenin en hızlı yolu.

**Dene:** `TOP_K = 3, 5, 10`.

In [ ]:
from paperlens.evals.retrieval_metrics import best_over_references
from paperlens.retrieval.retriever import Retriever

TOP_K = 5  # <- değiştir


def recall_for(method: str, top_k: int) -> float:
    retriever = Retriever(
        cfg.retrieval.model_copy(update={"method": method, "top_k": top_k}),
        bundle.dense,
        bundle.sparse,
    )
    values = []
    for q in sample:
        ranked = [sc.chunk.paragraph_ids for sc in retriever.retrieve(q.question, q.paper_id)]
        metrics = best_over_references(ranked, q.references, [top_k])
        if metrics:  # gold kanıtı olmayan (cevaplanamaz) sorular atlanır
            values.append(metrics[f"recall_at_{top_k}"])
    return float(np.mean(values))


for method in ("dense", "bm25", "hybrid"):
    print(f"{method:7} Recall@{TOP_K} = {recall_for(method, TOP_K):.3f}")

## 8 · Prompt

`pipelines/baseline.py` → `BaselinePipeline.build_prompt()` bulunan chunk'ları `[1] … [5]` diye numaralar ve `prompts/baseline_answer.md` şablonuna koyar. LLM makaleyi görmez; **sadece bu metni** görür.

In [ ]:
from paperlens.llm.client import OpenAICompatClient
from paperlens.pipelines.baseline import BaselinePipeline

retriever = Retriever(cfg.retrieval, bundle.dense, bundle.sparse)
llm = OpenAICompatClient(cfg.llm)
pipeline = BaselinePipeline(
    retriever,
    llm,
    cfg.generation.prompt_template.read_text(encoding="utf-8"),
    {p.paper_id: p.title for p in papers},
)

retrieved = retriever.retrieve(question.question, question.paper_id)
prompt = pipeline.build_prompt(question, retrieved)
print(prompt)
print(f"\n--- {len(prompt.split())} kelime")

## 9 · LLM (Ollama gerekir)

Önce aynı prompt'u **şema zorlamadan** gönderip modelin ham cevabını görelim; sonra pipeline'ın `structured_call()` ile JSON şemasına zorladığı gerçek akışı çalıştıralım.

In [ ]:
raw = llm.complete([{"role": "user", "content": prompt}])
print("HAM CEVAP:\n", raw.content)
print("\n", raw.usage)

In [ ]:
result = pipeline.run(question)  # retrieval + prompt + structured_call + atıf çözme
a = result.answer
print("tip       :", a.answer_type)
print("cevap     :", a.text)
print("atıf      :", a.citations)
print("kanıt     :", a.evidence_ids)
print("token/süre:", a.usage)

## 10 · Puanlama

`evals/records.py` → `build_record()` cevabı gold ile karşılaştırır. Answer-F1 normalize edilmiş kelimelerin örtüşmesidir (küçük harf, noktalama ve a/an/the atılır); birden fazla gold varsa en iyisi alınır.

In [ ]:
from paperlens.evals.answer_metrics import normalize_answer, token_f1
from paperlens.evals.records import build_record

rec = build_record(question, result, [1, 3, 5])
print("answer_f1          :", round(rec.answer_f1, 3))
print(
    "evidence_f1        :", round(rec.evidence_f1, 3), "| tavan:", round(rec.evidence_f1_ceiling, 3)
)
print("retrieval          :", {k: round(v, 3) for k, v in (rec.retrieval or {}).items()})
print("\nnormalize tahmin   :", normalize_answer(rec.prediction))
for r in question.references:
    print(
        f"gold ({r.answer_type:11}): {normalize_answer(r.answer_text)!r} -> F1 {token_f1(rec.prediction, r.answer_text):.3f}"
    )

In [ ]:
# Kendi tahminini puanla: F1'in uzun cevabı nasıl cezalandırdığını gör
MY_ANSWER = "10K image and caption pairs"  # <- değiştir
for r in question.references:
    print(f"{token_f1(MY_ANSWER, r.answer_text):.3f}  vs  {r.answer_text}")

## Bitti: sırada ne var

- `QUESTION_INDEX`'i değiştirip başa dön; özellikle F1'i 0 olan bir soru bul ve hatanın retrieval'da mı (Recall@5 = 0) yoksa LLM'de mi olduğunu teşhis et.
- 7. adımda `TOP_K` ve yöntemleri dene; sonra YAML'da değiştirip `uv run paperlens eval` ile LLM'li tam deneyi çalıştır.
- Aynı soruyu terminalde de izleyebilirsin: `uv run paperlens trace --index 16`.

In [ ]:
bundle.dense.close()  # Qdrant yerel dosya kilidini bırak